# Programação linear

A Programação Linear (PL) aprimora a resolução de sistemas de equações lineares ao incorporar uma função-objetivo a ser maximizada ou minimizada. Para $n$ variáveis e $m$ restrições, temos.

\begin{equation}
\text{max (or min)} \sum_{j=i}^{n} c_j x_j
\end{equation}

sujeito a:

\begin{equation}
\text{max (or min)} \sum_{j=i}^{n} a_ij x_j \leq (\text{ou =, ou} \geq) a_i \text{ para } i=1,...,m \text{ e } a_j \geq 0
\end{equation}


## Exemplo 1: Problema de alocação de terras

Uma propriedade apresenta dois talhões florestais aptos para corte: talhão A com $40 ha$ e $84 m^3/ha$ de madeira disponíveis; e talhão B com $18 ha$ e uma produção de $112 m^3/ha$. O custo por ha para a administração da venda de madeira é de R$ 300, e a disponibilidade de capital é de R$ 15.000. Ambos os talhões permitem o desenvolvimento de atividades recreacionistas. Anualmente, o talhão 1 é capaz de sustentar 480 visitantes por hectare e o talhão B apresenta capacidade para 1.920 visitantes por hectare. A propriedade deve ser capaz de receber no mínimo 10.000 visitantes/ano. Naturalmente, cada hectare cortado fica inutilizado para atividades de recreação. O problema é determinar quantos hectares explorar em cada talhão de forma a maximizar o volume de madeira cortada.

O problema pode ser resolvido da seguinte forma: 

O objetivo é maximizar o corte de madeira: $Max(Z) = 84 \times A + 112 \times B$
Alternativas: exploração dos talhões A e B. 
Restrições: 
1. Área finita dos talhões A e B: $A \leq 40$ e $B \leq 18$
2. Caixa disponível: $300 \times A + 300 \times B \leq 15.000$
3. Recreação: $400 \times (40-A) + 1920 \times (18-B) \geq 10.000$
4. Positividade das alternativas: $A, B \geq 0$

Chamando as alternativas de A e B, em que:
A = número de hectares a serem explorados no talhão A; 
B = número de hectares a serem explorados no talhão B. 

In [7]:
# Importar biblioteca
import pulp

# Instanciar o problema de otimização
modelo = pulp.LpProblem("Problema de maximização", pulp.LpMaximize)
A = pulp.LpVariable('A', lowBound=0, cat='Continuous')
B = pulp.LpVariable('B', lowBound=0, cat='Continuous')
# cat='Integer' se for necessário que as variáveis sejam inteiras


# Função objetivo
modelo += 84 * A + 112 * B, "Volume de madeira cortada"

# Restrições
modelo += A <= 40
modelo += B <= 18
modelo += 300*A+300*B <= 15000
modelo += 400*(40-A)+1920*(18-B) >= 10000
modelo += A>=0
modelo += B>=0

# Resolver o problema
modelo.solve()
pulp.LpStatus[modelo.status]

# Exibir os valores das variáveis de decisão
print("Talhão A = {}".format(A.varValue))
print("Talhão B = {}".format(B.varValue))

# Exibir o valor da função objetivo
print(pulp.value(modelo.objective))

Talhão A = 36.473684
Talhão B = 13.526316
4578.7368480000005


### Exibir gráfico

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definir os limites para A e B
valorA = np.linspace(0, 60, 500)  # Valor máximo de A é 40, mas estendendo um pouco para melhor visualização
valorB = np.linspace(0, 30, 500)  # Valor máximo de B é 18, mas estendendo um pouco para melhor visualização

A, B = np.meshgrid(valorA, valorB)

# Dfinir inequações para melhor visualização da região viável
# 1. A <= 40
# 2. B <= 18
# 3. 300*A + 300*B <= 15000  => A + B <= 50
# 4. 400*(40-A) + 1920*(18-B) >= 10000
#    16000 - 400*A + 34560 - 1920*B >= 10000
#    50560 - 400*A - 1920*B >= 10000
#    40560 >= 400*A + 1920*B
#    400*A + 1920*B <= 40560
# 5. A >= 0
# 6. B >= 0

# Create a boolean array for each constraint
restircao1 = (A <= 40)
restircao2 = (B <= 18)
restircao3 = (A + B <= 50)
restircao4 = (400 * A + 1920 * B <= 40560)
restircao5 = (A >= 0)
restircao6 = (B >= 0)

# Combine all constraints to find the feasible region
feasible_region = restircao1 & restircao2 & restircao3 & restircao4 & restircao5 & restircao6

# Plotar figura
plt.figure(figsize=(10, 8))

# Destacar a região viável
plt.imshow(feasible_region.astype(int), origin='lower',
           extent=[valorA.min(), valorA.max(), valorB.min(), valorB.max()],
           cmap='Greens', alpha=0.3, aspect='auto')

# Plot boundary lines
# A <= 40
plt.plot([40, 40], [0, max(valorB)], color='red', linestyle='--', label='A <= 40')
# B <= 18
plt.plot([0, max(valorA)], [18, 18], color='blue', linestyle='--', label='B <= 18')

# A + B <= 50 => B = 50 - A
plt.plot(valorA, 50 - valorA, color='purple', linestyle='--', label='A + B <= 50')

# 400*A + 1920*B <= 40560 => B = (40560 - 400*A) / 1920
plt.plot(valorA, (40560 - 400 * valorA) / 1920, color='orange', linestyle='--', label='400A + 1920B <= 40560')

# A >= 0
plt.axvline(x=0, color='gray', linestyle='--')
# B >= 0
plt.axhline(y=0, color='gray', linestyle='--')

# Adicionar o ponto ótimo encontrado
optimal_A = 36.473684
optimal_B = 13.526316
plt.plot(optimal_A, optimal_B, 'r*', markersize=15, label=f'Ponto Ótimo: (A={optimal_A:.2f}, B={optimal_B:.2f})')

plt.xlim(-1, 60)
plt.ylim(-1, 30)
plt.xlabel('Hectares no Talhão A')
plt.ylabel('Hectares no Talhão B')
plt.title('Região Viável para o Problema de Alocação de Terras')
plt.legend()
plt.grid(True)
plt.show()

## Formulação de ração de custo mínimo

Uma aplicação bem sucedida de programação linear diz respeito à formulação de dietas, na qual se deseja obter a ração de mínimo custo (ou mínimo preço), a partir da disponibilidade de uma série de alimentos, mas respeitando-se as exigências nutricionais pertinentes à idade e tipo de animal. Para a formulação de ração para suínos, são apresentados os seguintes dados:



| Item                  | Mínimo Requerido | Máximo Permitido |
|-----------------------|-----------------:|-----------------:|
| Proteina (%)          | 15,50            | 16,00            |
| Energia (kcal/kg)     | 3.260,00         | 3.360,00         |
| P (%)                 | 0,50             | 0,52             |
| Ca/P                  | 1,30             | 1,40             |
| Farelo de trigo (%)   | 0,00             | 15,00            |
| Farinha de carne (%)  | 0,00             | 3,00             |
| Farinha de sangue (%) | 0,00             | 2,00             |
| Lisina (%)            | 0,69             | sem limite       |
| Sal (%)               | 0,50             | 0,50             |
| Mistura A (%)         | 0,10             | 0,10             |
| Mistura B (%)         | 0,10             | 0,10             |

| Alimento          | Variável | Proteína (%) | Energia (kcal/kg) | Cálcio (%) | Fósforo (%) | Lisina (%) | Preço (R$/kg) |
|-------------------|----------|-------------:|------------------:|-----------:|------------:|-----------:|--------------:|
| Milho             | Milho    | 8,51         | 3.493             | 0,02       | 0,27        | 0,23       | 1,80          |
| Farinha de soja   | Fsoja    | 45,60        | 3.378             | 0,36       | 0,55        | 2,87       | 4,20          |
| Farinha de trigo  | Ftrigo   | 15,30        | 2.103             | 0,12       | 0,88        | 0,57       | 2,00          |
| Farinha de carne  | Fcarne   | 45,20        | 2.133             | 11,60      | 5,40        | 2,28       | 7,50          |
| Farinha de sangue | Fsan     | 80,90        | 2.809             | 0,20       | 0,15        | 6,57       | 9,00          |
| Fosfato bicálcico | Fosbica  | -            | -                 | 22,61      | 17,03       | -          | 14,50         |
| Cálcio            | Calca    | -            | -                 | 37,00      | -           | -          | 0,80          |
| Sal               | Sal      | -            | -                 | -          | -           | -          | 3,00          |
| Mistura mineral   | MistA    | -            | -                 | -          | -           | -          | 28,00         |
| Mist. Vitamínica  | MistB    | -            | -                 | -          | -           | -          | 145,00        |

Admitindo que um produtor deseja formular uma ração de custo mínimo para suínos em crescimento, deve-se determinar a melhor composição para 1 kg de ração. Assim, minimizando o custo total de 1 kg da mistura, a partir das informações fornecidas.

$Min C = 1,80 Milho + 4,20 Fsoja + 2,00 Ftrigo +7,50 Fcarne +9,00 Fsan + 14,50 Fosbica +0,80 Calca + 3,00 Sal + 28,00 MistA + 145,00 MistB$

sujeito a:


Proteína: $15,50 ≤ 8,51 Milho + 45,60 Fsoja + 15,30 Ftrigo + 45,20 Fcarne + 80,90 Fsan ≤ 16,00$

Energia": $3.260 3.493 Milho + 3.378 Fsoja + 2.103 Ftrigo + 2.133 Fcarne + 2.809 Fsan≤ 3.360$

Cálcio: $0,02 Milho +0,36 Fsoja + 0,12 Ftrigo + 11,60 Fcarne + 0,20 Fsan + 22,61 Fosbica + 37,00 Calca = Ca$

Fósforo: $0,27 Milho +0,55 Fsoja + 0,88 Ftrigo + 5,40 Fcarne + 0,15 Fsan + 17,03 Fosbica = Pe 0,0050 ≤ P ≤ 0,0052$

Lisina: $0,23 Milho + 2,87 Fsoja + 0,57 Ftrigo +2,28 Fcarne +6,57 Fsan ≥ 0,69$

Máximo de farinha de trigo: $Ftrigo ≤ 0,15$

Máximo de farinha de carne: $Fcarne ≤ 0,03$

Máximo de farinha de sangue: $Fsan≤ 0,02$

Sal: $Sal = 0,005$

Mistura A: $MistA = 0,001$

Mistura B: $MistB = 0,001$

Ca/P:

$Ca-1,30 P ≤ 0$

$Ca - 1,40 P ≥ 0$

In [ ]:
# Instalar pacote pulp
!pip install pulp

In [ ]:
# Importar biblioteca
import pulp

# Instanciar o problema de otimização
modelo = pulp.LpProblem("Problema de minimização", pulp.LpMinimize)
Milho = pulp.LpVariable('Milho', lowBound=0, cat='Continuous')
Fsoja = pulp.LpVariable('Fsoja', lowBound=0, cat='Continuous')
Ftrigo = pulp.LpVariable('Ftrigo', lowBound=0, cat='Continuous')
Fcarne = pulp.LpVariable('Fcarne', lowBound=0, cat='Continuous')
Fsan = pulp.LpVariable('Fsan', lowBound=0, cat='Continuous')
Fosbica = pulp.LpVariable('Fosbica', lowBound=0, cat='Continuous')
Calca = pulp.LpVariable('Calca', lowBound=0, cat='Continuous')
Sal = pulp.LpVariable('Sal', lowBound=0, cat='Continuous')
MistA = pulp.LpVariable('MistA', lowBound=0, cat='Continuous')
MistB = pulp.LpVariable('MistB', lowBound=0, cat='Continuous')
# cat='Integer' se for necessário que as variáveis sejam inteiras


# Função objetivo
modelo += 1.80*Milho + 4.20*Fsoja + 2.00*Ftrigo +7.50*Fcarne +9.00*Fsan + 14.50*Fosbica +0.80*Calca + 3.00*Sal + 28.00*MistA + 145.00*MistB

# Restrições
modelo += 
modelo += 15.50 <= 8.51*Milho + 45.60*Fsoja + 15.30*Ftrigo + 45.20*Fcarne + 80.90*Fsan <= 16.00
modelo += 3.260 <= 3.493*Milho + 3.378*Fsoja + 2.103*Ftrigo + 2.133*Fcarne + 2.809*Fsan<= 3.360
modelo += 0.02*Milho +0.36*Fsoja + 0.12*Ftrigo + 11.60*Fcarne + 0.20*Fsan + 22.61*Fosbica + 37.00*Calca = Ca
modelo += 0.27*Milho +0.55*Fsoja + 0.88*Ftrigo + 5.40*Fcarne + 0.15*Fsan + 17.03*Fosbica = P
modelo += 0.0050 <= P <= 0.0052
modelo += 0.23*Milho + 2.87*Fsoja + 0.57*Ftrigo +2.28*Fcarne +6.57*Fsan ≥ 0.69
modelo += Ftrigo <= 0.15
modelo += Fcarne <= 0.03
modelo += Fsan<= 0.02
modelo += Sal = 0.005
modelo += MistA = 0.001
modelo += MistB = 0.001
modelo += Ca-1.30*P <= 0
modelo += Ca-1.40*P ≥ 0

# Resolver o problema
modelo.solve()
pulp.LpStatus[modelo.status]

# Exibir os valores das variáveis de decisão
print("Talhão A = {}".format(A.varValue))
print("Talhão B = {}".format(B.varValue))

# Exibir o valor da função objetivo
print(pulp.value(modelo.objective))